# Lesson 6: Spectral analysis

Neurocampus course "Signals of the whole brain"

Daria Kleeva

dkleeva@gmail.com

April 15, 2026


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox
 
plt.rcParams.update({
    'figure.figsize': (12, 4),
    'font.size': 13,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## Properties of the signal

A sinusoid has exactly three knobs: **frequency, amplitude, phase**. Move the sliders: see what each one does.

In [ ]:
t = np.linspace(0, 2, 1000) 
 
def plot_single_sine(freq=2.0, amp=1.0, phase=0.0):
    signal = amp * np.sin(2 * np.pi * freq * t + phase)
    plt.figure(figsize=(12, 4))
    plt.plot(t, signal, color='steelblue', linewidth=2)
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.title(f'Sinusoid: frequency = {freq} Hz, amplitude = {amp}, phase = {phase:.1f} rad')
    plt.ylim(-3.5, 3.5)
    plt.axhline(0, color='gray', linewidth=0.5)
    plt.tight_layout()
    plt.show()
 
interact(
    plot_single_sine,
    freq=FloatSlider(min=0.5, max=10, step=0.5, value=2, description='Freq (Hz)'),
    amp=FloatSlider(min=0.1, max=3, step=0.1, value=1, description='Amplitude'),
    phase=FloatSlider(min=0, max=2*np.pi, step=0.1, value=0, description='Phase (rad)'),
);

Let's take a few sinusoids at different frequencies and add them up.

In [ ]:
components = [
    (2,  1.0, 0),
    (5,  0.6, np.pi / 4),
    (11, 0.3, np.pi / 2),
]

t = np.linspace(0, 2, 1000)
fig, axes = plt.subplots(len(components) + 1, 1, figsize=(12, 3 * (len(components) + 1)))
 
sum_signal = np.zeros_like(t)
colors = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']
 
for i, (freq, amp, phase) in enumerate(components):
    wave = amp * np.sin(2 * np.pi * freq * t + phase)
    sum_signal += wave
    axes[i].plot(t, wave, color=colors[i], linewidth=2)
    axes[i].set_title(f'Component {i + 1}: {freq} Hz, amplitude {amp}', fontsize=12)
    axes[i].set_ylim(-2.5, 2.5)
    axes[i].set_ylabel('Amplitude')
 
# Sum
axes[-1].plot(t, sum_signal, color='black', linewidth=2)
axes[-1].set_title('Sum of all components', fontsize=12, fontweight='bold')
axes[-1].set_xlabel('Time (s)')
axes[-1].set_ylabel('Amplitude')
 
plt.tight_layout()
plt.show()
 

**Question:** 

Looking at the bottom plot, can you tell which frequencies are in there? You can't: and that's exactly why we need the Fourier Transform.

## Approximating the square wave

A square wave is sharp, with hard edges. Can we build it from sinusoids? 

In [ ]:
def plot_square_wave_approx(n_harmonics=1):
    t = np.linspace(0, 2, 2000)
    f0 = 1  # fundamental frequency
 
    # Target square wave
    square = np.sign(np.sin(2 * np.pi * f0 * t))
 
    # Approximation: sum of odd harmonics
    approx = np.zeros_like(t)
    for k in range(n_harmonics):
        n = 2 * k + 1  # 1, 3, 5, 7, ...
        approx += (4 / (np.pi * n)) * np.sin(2 * np.pi * n * f0 * t)
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
    axes[0].plot(t, square, '--', color='gray', linewidth=1.5, label='Square wave')
    axes[0].plot(t, approx, color='crimson', linewidth=2.5,
                 label=f'Approximation ({n_harmonics} harmonics)')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Amplitude')
    axes[0].set_title('Time domain')
    axes[0].legend(loc='upper right')
    axes[0].set_ylim(-1.6, 1.6)
 
    harmonics = [2 * k + 1 for k in range(n_harmonics)]
    amplitudes = [4 / (np.pi * n) for n in harmonics]
    axes[1].bar(harmonics, amplitudes, color='crimson', alpha=0.7, width=0.6)
    axes[1].set_xlabel('Harmonic number')
    axes[1].set_ylabel('Amplitude')
    axes[1].set_title('The "recipe": amplitude of each harmonic')
    axes[1].set_xlim(0, 32)
    axes[1].set_ylim(0, 1.5)
 
    plt.tight_layout()
    plt.show()
 
interact(
    plot_square_wave_approx,
    n_harmonics=IntSlider(min=1, max=30, step=1, value=1, description='Harmonics'),
);
 

**Takeaway:** *any* signal can be represented as a sum of sinusoids. The only question is — which ones and in what proportions.

Why do we use only odd harmonics? This is a property of the square wave's symmetry. A square wave has half-wave symmetry: if you shift it by half a period,  it flips upside down: f(t + T/2) = −f(t).

Odd harmonics (1f, 3f, 5f...) share this property — shift them by half a period and they flip too. Even harmonics (2f, 4f, 6f...) don't — they repeat themselves instead of flipping. So they simply can't contribute to a signal that has this symmetry. They cancel out to zero.

In [ ]:
def plot_even_odd_demo(use_odd=True, use_even=False, n_harmonics=15):
    t = np.linspace(0, 2, 2000)
    f0 = 1
 
    square = np.sign(np.sin(2 * np.pi * f0 * t))
 
    approx = np.zeros_like(t)
    bar_ns = []
    bar_amps = []
    bar_cols = []
 
    for n in range(1, n_harmonics + 1):
        is_odd = (n % 2 == 1)
        if is_odd and not use_odd:
            continue
        if not is_odd and not use_even:
            continue
        coeff = 4 / (np.pi * n)
        approx += coeff * np.sin(2 * np.pi * n * f0 * t)
        bar_ns.append(n)
        bar_amps.append(coeff)
        bar_cols.append('#2196F3' if is_odd else '#FF9800')
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
    axes[0].plot(t, square, '--', color='gray', linewidth=1.5, label='Square wave')
    axes[0].plot(t, approx, color='crimson', linewidth=2.5, label='Approximation')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Amplitude')
    parts = []
    if use_odd:
        parts.append('odd')
    if use_even:
        parts.append('even')
    label = ' + '.join(parts) if parts else 'none'
    axes[0].set_title(f'Harmonics used: {label}')
    axes[0].legend(loc='upper right')
    axes[0].set_ylim(-1.8, 1.8)
 
    if bar_ns:
        axes[1].bar(bar_ns, bar_amps, color=bar_cols, alpha=0.7, width=0.6)
    axes[1].set_xlabel('Harmonic number')
    axes[1].set_ylabel('Amplitude')
    axes[1].set_title('Blue = odd, Orange = even')
    axes[1].set_xlim(0, n_harmonics + 1)
    axes[1].set_ylim(0, 1.5)
 
    plt.tight_layout()
    plt.show()
 
interact(
    plot_even_odd_demo,
    use_odd=Checkbox(value=True, description='Odd harmonics'),
    use_even=Checkbox(value=False, description='Even harmonics'),
    n_harmonics=IntSlider(min=1, max=30, step=1, value=15, description='Up to N'),
);

You may have noticed: no matter how many harmonics we add, the approximation always overshoots near the sharp edges of the square wave. Those little "horns" at the transitions don't go away — they get narrower, but their height stays about the same (~9% overshoot). This is the **Gibbs phenomenon**. It's a fundamental property of  Fourier series at discontinuities. Smooth sinusoids simply cannot reproduce an instantaneous jump perfectly — they always ring around it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
 
t = np.linspace(0, 2, 5000)
f0 = 1
square = np.sign(np.sin(2 * np.pi * f0 * t))
 
for idx, n_harm in enumerate([5, 25, 100]):
    approx = np.zeros_like(t)
    for k in range(n_harm):
        n = 2 * k + 1
        approx += (4 / (np.pi * n)) * np.sin(2 * np.pi * n * f0 * t)
 
    axes[idx].plot(t, square, '--', color='gray', linewidth=1, label='Target')
    axes[idx].plot(t, approx, color='crimson', linewidth=1.5, label='Approx')
    axes[idx].set_title(f'{n_harm} harmonics')
    axes[idx].set_xlim(0.4, 0.6)   # zoom into the rising edge
    axes[idx].set_ylim(-1.5, 1.5)
    axes[idx].axhline(1.0, color='black', linewidth=0.5, linestyle=':')
    axes[idx].axhline(-1.0, color='black', linewidth=0.5, linestyle=':')
    axes[idx].set_xlabel('Time (s)')
    if idx == 0:
        axes[idx].set_ylabel('Amplitude')
        axes[idx].legend(fontsize=10)
 
plt.suptitle('Gibbs phenomenon: the overshoot never disappears, only gets narrower',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## The reverse: from signal to recipe

We know how to build a signal from sinusoids. But what if someone hands us a complex signal — can we figure out which frequencies are inside?  That's exactly what the **Fourier Transform** does.

In [ ]:
fs = 500          # sampling rate (Hz)
duration = 2      # seconds
t = np.arange(0, duration, 1 / fs)
 
freq_true = [2, 5, 11]
amp_true  = [1.0, 0.6, 0.3]
 
signal = np.zeros_like(t)
for f, a in zip(freq_true, amp_true):
    signal += a * np.sin(2 * np.pi * f * t)
 

In [ ]:
N = len(t)
fft_vals = np.fft.fft(signal)
fft_freqs = np.fft.fftfreq(N, d=1 / fs)

In [ ]:
plt.plot(fft_freqs)

What are the negative frequencies? It's a mathematical consequence of how the Fourier Transform works.

A real-valued sinusoid `sin(2π·f·t)` can be written as a sum of two complex exponentials — one spinning at +f and one spinning at −f. The practical takeaway is simple:** for real-valued signals (like EEG), the negative half is always a mirror image of the positive half. It carries no extra information. So we throw it away and multiply by 2 to keep the correct amplitude. That's what the code below does.

In [ ]:
all_freqs = fft_freqs
all_amps = (1 / N) * np.abs(fft_vals)  # note: 1/N, not 2/N — we haven't discarded the mirror yet

In [ ]:
all_freqs = fft_freqs
all_amps = (1 / N) * np.abs(fft_vals)  # note: 1/N, not 2/N — we haven't discarded the mirror yet
 
# Sort for nicer plotting (fftfreq returns [0, +, ..., -, ...])
sort_idx = np.argsort(all_freqs)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
axes[0].plot(all_freqs[sort_idx], all_amps[sort_idx], color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Full (two-sided) FFT — notice the mirror symmetry')
axes[0].set_xlim(-20, 20)
axes[0].axvline(0, color='gray', linewidth=0.5, linestyle=':')
for f in freq_true:
    axes[0].annotate(f'+{f}', xy=(f, 0.5), fontsize=10, ha='center', color='crimson')
    axes[0].annotate(f'−{f}', xy=(-f, 0.5), fontsize=10, ha='center', color='crimson')
 
# --- One-sided: discard the mirror, multiply by 2 ---
pos_mask = fft_freqs >= 0
freqs = fft_freqs[pos_mask]
amplitudes = (2 / N) * np.abs(fft_vals[pos_mask])
 
axes[1].plot(freqs, amplitudes, color='steelblue', linewidth=1.5)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('One-sided spectrum — this is what we actually use')
axes[1].set_xlim(0, 20)
for f, a in zip(freq_true, amp_true):
    axes[1].annotate(f'{f} Hz', xy=(f, a), fontsize=11, fontweight='bold',
                     ha='center', va='bottom', color='crimson')
 
plt.tight_layout()
plt.show()

The Fourier Transform recovered exactly the frequencies and amplitudes we put in! What if the signal is noisy?

In [ ]:
np.random.seed(42)
noise = np.random.randn(len(t)) * 0.8  
noisy_signal = signal + noise
 
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
axes[0].plot(t, noisy_signal, color='black', linewidth=0.8, alpha=0.8)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Noisy signal — try to spot the rhythms by eye')
 
# FFT of the noisy signal
fft_noisy = np.fft.fft(noisy_signal)
amp_noisy = (2 / N) * np.abs(fft_noisy[pos_mask])
 
axes[1].plot(freqs, amp_noisy, color='steelblue', linewidth=1, alpha=0.7)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('FFT of the noisy signal — peaks are still visible!')
axes[1].set_xlim(0, 20)
 
for f in freq_true:
    axes[1].axvline(f, color='crimson', linestyle='--', alpha=0.5)
 
plt.tight_layout()
plt.show()

The peaks are still there, but the spectrum is "hairy" because of the noise. This is exactly why, for real data, we use **Welch's method** — it averages many short FFTs and produces a much cleaner spectrum.

## Welch's method

The core issue: a single FFT of N points gives N/2 frequency bins, but each bin is estimated from just one realization of the signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
for i, seed in enumerate([42, 123]):
    np.random.seed(seed)
    noisy = signal + np.random.randn(len(t)) * 5
    fft_n = np.fft.fft(noisy)
    amp_n = (2 / N) * np.abs(fft_n[pos_mask])
    axes[i].plot(freqs, amp_n, color='steelblue', linewidth=0.8, alpha=0.7)
    axes[i].set_title(f'FFT — noise realization {i + 1}')
    axes[i].set_xlabel('Frequency (Hz)')
    axes[i].set_ylabel('Amplitude')
    axes[i].set_xlim(0, 20)
    for f in freq_true:
        axes[i].axvline(f, color='crimson', linestyle='--', alpha=0.4)
 
plt.suptitle('Same signal, different noise — the spectra look quite different',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

The peaks are roughly in the same place, but the overall shape jumps around. We need a way to reduce the variance of the spectrum estimate. The solution:

1. **Crop** the signal into shorter overlapping segments.
2. **Window** each segment (taper the edges to reduce spectral leakage).
3. **FFT** each segment and compute its power spectrum.
4. **Average** all the spectra together.

In [ ]:
from scipy.signal import welch as scipy_welch
 
fs = 500
duration_long = 100  
t_long = np.arange(0, duration_long, 1 / fs)
 
signal_long = np.zeros_like(t_long)
for f, a in zip(freq_true, amp_true):
    signal_long += a * np.sin(2 * np.pi * f * t_long)
 
np.random.seed(42)
noisy_long = signal_long + np.random.randn(len(t_long)) * 5

Below we manually split the signal into segments, compute FFT of each, and watch the average get smoother as we include more segments.

In [ ]:
seg_len = 1 * fs  # 1-second segments (= 1 Hz frequency resolution)
overlap = seg_len // 2  # 50% overlap
step = seg_len - overlap

In [ ]:
segments = []
start = 0
while start + seg_len <= len(noisy_long):
    segments.append(noisy_long[start:start + seg_len])
    start += step
 
n_segments = len(segments)

In [ ]:
from scipy.signal import get_window
window = get_window('hann', seg_len)
seg_freqs = np.fft.rfftfreq(seg_len, d=1 / fs)
all_spectra = []
 
for seg in segments:
    windowed = seg * window
    fft_seg = np.fft.rfft(windowed)
    # Power spectrum (V²/Hz) — proper normalization for Welch
    psd_seg = (2 / (fs * np.sum(window ** 2))) * np.abs(fft_seg) ** 2
    all_spectra.append(psd_seg)
 
all_spectra = np.array(all_spectra)
 

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
n_to_show = [1, 3, 7, n_segments]
 
for idx, n_avg in enumerate(n_to_show):
    avg_psd = np.mean(all_spectra[:n_avg], axis=0)
    axes[idx].semilogy(seg_freqs, avg_psd, color='steelblue', linewidth=1.2)
    axes[idx].set_title(f'Average of {n_avg} segment{"s" if n_avg > 1 else ""}')
    axes[idx].set_xlabel('Frequency (Hz)')
    axes[idx].set_xlim(0, 25)
    for f in freq_true:
        axes[idx].axvline(f, color='crimson', linestyle='--', alpha=0.4, linewidth=0.8)
    if idx == 0:
        axes[idx].set_ylabel('Power (log scale)')
 
plt.suptitle('Welch step by step: more segments averaged → cleaner spectrum',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
 

You don't need to do this manually. SciPy does it in one line. The key parameter is `nperseg` — the length of each segment in samples.

In [ ]:
fft_raw = np.fft.rfft(noisy_long)
freqs_raw = np.fft.rfftfreq(len(noisy_long), d=1 / fs)
psd_raw = (2 / (fs * len(noisy_long))) * np.abs(fft_raw) ** 2
 

In [ ]:
freqs_welch, psd_welch = scipy_welch(noisy_long, fs=fs, nperseg=1*fs, noverlap=fs//2)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
axes[0].semilogy(freqs_raw, psd_raw, color='steelblue', linewidth=0.5, alpha=0.6)
axes[0].set_title('Raw FFT power spectrum')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Power (log scale)')
axes[0].set_xlim(0, 25)
for f in freq_true:
    axes[0].axvline(f, color='crimson', linestyle='--', alpha=0.4)
 
axes[1].semilogy(freqs_welch, psd_welch, color='steelblue', linewidth=1.5)
axes[1].set_title('Welch method (1-second segments)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_xlim(0, 25)
for f in freq_true:
    axes[1].axvline(f, color='crimson', linestyle='--', alpha=0.4)
 
plt.suptitle('Raw FFT vs Welch — same data, huge difference in readability',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

The only parameter you really need to think about is `nperseg` — the segment length.

- Longer segments → better frequency resolution but fewer segments to average → noisier spectrum.
- Shorter segments → smoother spectrum but worse frequency resolution (close peaks merge together).

The frequency resolution is: Δf = fs / nperseg = 1 / (segment duration in seconds). So a 1-second segment gives 1 Hz resolution, a 2-second segment gives 0.5 Hz, etc.
 Let's see this tradeoff in action.

In [ ]:
def plot_welch_tradeoff(seg_duration_s=1.0):
    nperseg = int(seg_duration_s * fs)
    if nperseg > len(noisy_long):
        nperseg = len(noisy_long)
    noverlap = nperseg // 2
    n_segs = max(1, int((len(noisy_long) - noverlap) / (nperseg - noverlap)))
    freq_res = fs / nperseg
 
    f_w, psd_w = scipy_welch(noisy_long, fs=fs, nperseg=nperseg, noverlap=noverlap)
 
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.semilogy(f_w, psd_w, color='steelblue', linewidth=1.5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Power (log scale)')
    ax.set_title(
        f'Segment = {seg_duration_s:.1f} s → '
        f'Δf = {freq_res:.2f} Hz, '
        f'~{n_segs} segments averaged'
    )
    ax.set_xlim(0, 25)
    for f in freq_true:
        ax.axvline(f, color='crimson', linestyle='--', alpha=0.4)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
 
interact(
    plot_welch_tradeoff,
    seg_duration_s=FloatSlider(min=0.25, max=5.0, step=0.25, value=1.0,
                               description='Segment (s)'),
);

Rule of thumb for EEG: segment length of 2–4 seconds is a common starting point. It gives 0.25–0.5 Hz resolution with enough segments for a reasonably smooth spectrum.

What is **window function**? When we chop the signal into segments, each segment starts and ends abruptly — the edges don't match up. This creates spectral leakage: energy from a peak "leaks" into neighboring frequencies.

A window function (like the Hann window) gently tapers the segment edges to zero, reducing this leakage. SciPy's `welch` uses Hann by default.

## Multitapers

Welch reduces variance by chopping in time. But chopping has a cost: each segment is short, so frequency resolution suffers. **Multitapers** take a completely different approach: instead of chopping the signal, they apply multiple different windows (tapers) to the *entire* signal, compute the spectrum for each taper, and average. You get variance reduction without losing any time points.

The tapers used are called **DPSS (Discrete Prolate Spheroidal Sequences)**, also known as **Slepian sequences**. They are mathematically designed to concentrate maximum energy within a chosen frequency bandwidth.

In [ ]:
from scipy.signal.windows import dpss
 
# --- Show what DPSS tapers look like ---
N_seg = 2 * fs  # 2-second window
NW = 4          # time-bandwidth product (we'll explain below)
n_tapers = 2 * NW - 1  # conventional choice: 2*NW - 1 tapers
 
tapers = dpss(N_seg, NW, Kmax=min(n_tapers, 7))  # show up to 7
t_seg = np.arange(N_seg) / fs
 
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
 
colors_tap = plt.cm.viridis(np.linspace(0.1, 0.9, len(tapers)))
for i, (tap, col) in enumerate(zip(tapers, colors_tap)):
    axes[0].plot(t_seg, tap, color=col, linewidth=1.5, label=f'Taper {i}')
axes[0].set_title(f'DPSS tapers (NW = {NW}, showing {len(tapers)} tapers)')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].legend(fontsize=9, ncol=4, loc='upper right')
 
# Show how each taper "sees" a different version of the signal
seg_example = noisy_long[:N_seg]
for i, (tap, col) in enumerate(zip(tapers[:4], colors_tap[:4])):
    tapered_sig = seg_example * tap
    axes[1].plot(t_seg, tapered_sig + i * 4, color=col, linewidth=1,
                 label=f'Signal × Taper {i}')
axes[1].set_title('Same signal viewed through different tapers (offset for clarity)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude (offset)')
axes[1].legend(fontsize=9, ncol=4, loc='upper right')
 
plt.tight_layout()
plt.show()
 

Each taper is different, but they are all orthogonal — their dot products are zero. When we multiply the signal by each taper and compute the FFT, we get independent spectral estimates that we can average.

In Welch the key knob was `nperseg` (segment length). In multitapers the key knob is **NW** (time–bandwidth product, also called the half-bandwidth parameter). 
 - NW controls how many tapers you can use: K = 2·NW − 1 tapers.
 - More tapers → more averaging → smoother spectrum, but each taper "smears" frequencies over a bandwidth of ±NW/T Hz (where T is the data length in seconds).
 - So higher NW = smoother but lower frequency resolution. 

In [ ]:
def plot_multitaper_NW(NW=4.0):
    N_mt = len(noisy_long)
    K = max(1, int(2 * NW - 1))
    T = N_mt / fs
    half_bw = NW / T
 
    tapers_mt = dpss(N_mt, NW, Kmax=K)
 
    psd_mt = np.zeros(N_mt // 2 + 1)
    for tap in tapers_mt:
        fft_tap = np.fft.rfft(noisy_long * tap)
        psd_mt += np.abs(fft_tap) ** 2
    psd_mt /= K

    psd_mt *= 2 / (fs * N_mt)
    f_mt = np.fft.rfftfreq(N_mt, d=1 / fs)
 
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.semilogy(f_mt, psd_mt, color='steelblue', linewidth=1.5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Power (log scale)')
    ax.set_title(
        f'Multitaper PSD — NW = {NW:.1f}, '
        f'K = {K} tapers, '
        f'half-bandwidth = ±{half_bw:.2f} Hz'
    )
    ax.set_xlim(0, 25)
    for f in freq_true:
        ax.axvline(f, color='crimson', linestyle='--', alpha=0.4)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
 
interact(
    plot_multitaper_NW,
    NW=FloatSlider(min=1.5, max=8, step=0.5, value=4, description='NW'),
);


In [ ]:
f_w, psd_w = scipy_welch(noisy_long, fs=fs, nperseg=2*fs, noverlap=fs)
 
NW = 4
K = 2 * NW - 1
N_mt = len(noisy_long)
tapers_mt = dpss(N_mt, NW, Kmax=K)
 
psd_mt = np.zeros(N_mt // 2 + 1)
for tap in tapers_mt:
    fft_tap = np.fft.rfft(noisy_long * tap)
    psd_mt += np.abs(fft_tap) ** 2
psd_mt /= K
psd_mt *= 2 / (fs * N_mt)
f_mt = np.fft.rfftfreq(N_mt, d=1 / fs)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
axes[0].semilogy(f_w, psd_w, color='steelblue', linewidth=1.5)
axes[0].set_title('Welch (2 s segments)')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Power (log scale)')
axes[0].set_xlim(0, 25)
 
axes[1].semilogy(f_mt, psd_mt, color='steelblue', linewidth=1.5)
axes[1].set_title('Multitaper (NW = 4, 7 tapers)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_xlim(0, 25)
 
for ax in axes:
    for f in freq_true:
        ax.axvline(f, color='crimson', linestyle='--', alpha=0.4)
 
plt.suptitle('Welch vs Multitaper on the same 10-second noisy signal',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Non-sinusoidal rhythms

Everything we've done so far assumes that brain oscillations are nice sinusoids. But real neural rhythms are often non-sinusoidal — they have asymmetric shapes with sharper peaks and broader troughs (or vice versa).


The most famous example is the **mu rhythm** (~10 Hz, sensorimotor cortex). Its waveform has a characteristic arc-like shape. And this shape creates a serious problem for Fourier-based analysis.

When you compute the FFT of a non-sinusoidal 10 Hz oscillation, you don't just get a peak at 10 Hz. You also get peaks at 20 Hz, 30 Hz, ... — the harmonics. These are not real beta or gamma oscillations. They are the artifacts of forcing a non-sinusoidal shape into a basis of pure sinusoids.

Let's see this happen.

In [ ]:
fs = 500
duration = 10
t_mu = np.arange(0, duration, 1 / fs)
 
f_mu = 10  # mu frequency
phase_mu = 2 * np.pi * f_mu * t_mu
 
mu_clean = np.sin(phase_mu - 0.3*np.sin(phase_mu))
 
sin_clean = np.sin(phase_mu)
 
np.random.seed(42)
noise_mu = np.random.randn(len(t_mu)) * 0
mu_noisy = mu_clean + noise_mu
sin_noisy = sin_clean + noise_mu
 

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
 
axes[0].plot(t_mu[:fs*2], sin_noisy[:fs*2], color='steelblue', linewidth=1.5)
axes[0].set_title('Pure sinusoidal 10 Hz (what Fourier assumes)')
axes[0].set_ylabel('Amplitude')
 
axes[1].plot(t_mu[:fs*2], mu_noisy[:fs*2], color='crimson', linewidth=1.5)
axes[1].set_title('Arc-shaped mu rhythm at 10 Hz (what the brain actually produces)')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Time (s)')
 
plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import welch as scipy_welch
 
# --- Welch spectra of both ---
f_sin, psd_sin = scipy_welch(sin_noisy, fs=fs, nperseg=2*fs)
f_arc, psd_arc = scipy_welch(mu_noisy, fs=fs, nperseg=2*fs)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
axes[0].semilogy(f_sin, psd_sin, color='steelblue', linewidth=1.5)
axes[0].set_title('Spectrum of pure sinusoidal 10 Hz')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Power (log scale)')
axes[0].set_xlim(0, 40)
axes[0].axvspan(8, 12, alpha=0.15, color='blue', label='Alpha band')
axes[0].axvspan(13, 30, alpha=0.15, color='orange', label='Beta band')
axes[0].legend(fontsize=9)
 
axes[1].semilogy(f_arc, psd_arc, color='crimson', linewidth=1.5)
axes[1].set_title('Spectrum of arc-shaped mu at 10 Hz')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_xlim(0, 40)
axes[1].axvspan(8, 12, alpha=0.15, color='blue', label='Alpha band')
axes[1].axvspan(13, 30, alpha=0.15, color='orange', label='Beta band')
axes[1].annotate('← Real peak (10 Hz)', xy=(10, psd_arc[np.argmin(np.abs(f_arc - 10))]),
                 fontsize=11, ha='left', va='bottom', color='black',
                 xytext=(15, psd_arc[np.argmin(np.abs(f_arc - 10))] * 2),
                 arrowprops=dict(arrowstyle='->', color='black'))
axes[1].annotate('← FALSE peak (20 Hz)\n    harmonic artifact!',
                 xy=(20, psd_arc[np.argmin(np.abs(f_arc - 20))]),
                 fontsize=11, ha='left', va='bottom', color='red',
                 xytext=(24, psd_arc[np.argmin(np.abs(f_arc - 20))] * 3),
                 arrowprops=dict(arrowstyle='->', color='red'))
axes[1].legend(fontsize=9)
 
plt.suptitle('Same 10 Hz rhythm, different shape → very different spectrum',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**EMD** is a completely different approach to decomposing signals. Unlike Fourier, it does not assume sinusoidal basis functions. Instead, it adaptively extracts **Intrinsic Mode Functions (IMFs)** — components that can be non-sinusoidal and whose frequency can change over time.

The algorithm (Huang et al., 1998) is iterative:
1. Find all local maxima and minima of the signal.
2. Fit envelopes through the maxima (upper) and minima (lower).
3. Compute the mean of the envelopes and subtract it from the signal.
4. Repeat until the result is an IMF (roughly symmetric around zero).
5. Subtract this IMF from the original and repeat to find the next IMF.

The result is a set of IMFs — from high frequency to low frequency — that together reconstruct the original signal.

In [ ]:
# ! pip3 install emd

In [ ]:
import emd
 

imfs = emd.sift.mask_sift(mu_noisy, max_imfs=8)
print(f'EMD found {imfs.shape[1]} IMFs')
 
best_imf_idx = None
best_power_10 = 0
imf_spectra = []
 
for i in range(imfs.shape[1]):
    fw, pw = scipy_welch(imfs[:, i], fs=fs, nperseg=2 * fs)
    imf_spectra.append((fw, pw))
    mask_10 = (fw >= 8) & (fw <= 12)
    p10 = np.sum(pw[mask_10])
    if p10 > best_power_10:
        best_power_10 = p10
        best_imf_idx = i
 
print(f'Dominant mu IMF: IMF {best_imf_idx}')
 

In [ ]:
imf_indices = []
for i in range(imfs.shape[1]):
    fw, pw = imf_spectra[i]
    peak_f = fw[np.argmax(pw)]
    if peak_f > 40 and len(imf_indices) == 0:
        imf_indices.append(i)  # a noise IMF
    elif 15 < peak_f < 25 and np.max(pw) > 0.005:
        imf_indices.append(i)  # harmonic IMF
    elif i == best_imf_idx:
        imf_indices.append(i)  # mu IMF
    elif peak_f < 8 and len(imf_indices) >= 3 and len(imf_indices) < 4:
        imf_indices.append(i)  # low-freq IMF
 
# Ensure we have at least the mu IMF
if best_imf_idx not in imf_indices:
    imf_indices.append(best_imf_idx)
 
n_show = len(imf_indices)
fig, axes = plt.subplots(n_show, 2, figsize=(14, 3 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]
 
t_plot = t_mu[:2 * fs]  # 2 seconds
 
for row, idx in enumerate(imf_indices):
    fw, pw = imf_spectra[idx]
    peak_f = fw[np.argmax(pw)]
 
    # Time domain
    axes[row, 0].plot(t_plot, imfs[:2 * fs, idx], color='steelblue', linewidth=1.5)
    label = ''
    if idx == best_imf_idx:
        label = ' ← MU RHYTHM'
        axes[row, 0].plot(t_plot, imfs[:2 * fs, idx], color='crimson', linewidth=1.5)
    axes[row, 0].set_title(f'IMF {idx}{label}', fontsize=12,
                           fontweight='bold' if idx == best_imf_idx else 'normal')
    axes[row, 0].set_ylabel('Amplitude')
    if row == n_show - 1:
        axes[row, 0].set_xlabel('Time (s)')
 
    # Frequency domain
    col = 'crimson' if idx == best_imf_idx else 'steelblue'
    axes[row, 1].semilogy(fw, pw, color=col, linewidth=1.5)
    axes[row, 1].set_xlim(0, 40)
    axes[row, 1].set_title(f'IMF {idx} spectrum — peak at {peak_f:.0f} Hz', fontsize=12)
    axes[row, 1].set_ylabel('Power')
    if row == n_show - 1:
        axes[row, 1].set_xlabel('Frequency (Hz)')
 
plt.suptitle('EMD decomposes the signal into Intrinsic Mode Functions',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

 Now we apply the **Hilbert transform** to the mu IMF to get its **instantaneous frequency** — the frequency at each moment in time. 

If the 20 Hz peak in the Fourier spectrum were a real beta oscillation,
we would expect to see moments where the instantaneous frequency
jumps to 20 Hz. If it's just a waveform shape artifact, the instantaneous frequency should stay around 10 Hz.

In [ ]:
from scipy.signal import hilbert as scipy_hilbert
 
mu_imf = imfs[:, best_imf_idx]
 

analytic = scipy_hilbert(mu_imf)
inst_phase = np.unwrap(np.angle(analytic))
inst_freq = np.diff(inst_phase) * fs / (2 * np.pi)
 
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
 

axes[0].plot(t_mu[:3*fs], mu_imf[:3*fs], color='crimson', linewidth=1.5)
axes[0].set_title('Mu rhythm IMF — note the non-sinusoidal shape')
axes[0].set_ylabel('Amplitude')
 

inst_amp = np.abs(analytic)
axes[1].plot(t_mu[:3*fs], inst_amp[:3*fs], color='darkorange', linewidth=1.5)
axes[1].set_title('Instantaneous amplitude (Hilbert envelope)')
axes[1].set_ylabel('Amplitude')
 

axes[2].plot(t_mu[:3*fs-1], inst_freq[:3*fs-1], color='steelblue', linewidth=1, alpha=0.7)
axes[2].axhline(10, color='crimson', linestyle='--', linewidth=1.5, label='10 Hz')
axes[2].axhline(20, color='orange', linestyle='--', linewidth=1.5, label='20 Hz (beta)')
axes[2].set_title('Instantaneous frequency — stays around 10 Hz, NOT 20 Hz')
axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylim(0, 30)
axes[2].legend(loc='upper right')
 
plt.suptitle('Hilbert–Huang Transform of the mu IMF',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
 
print(f'Instantaneous frequency — median: {np.median(inst_freq):.1f} Hz, '
      f'std: {np.std(inst_freq):.1f} Hz')

The instantaneous frequency hovers tightly around 10 Hz. There is no jump to 20 Hz — confirming that the "beta peak" in the Fourier spectrum was entirely a harmonic artifact of the non-sinusoidal waveform shape.

This is the **Hilbert–Huang Transform (HHT)** : EMD + Hilbert. 

## Periodic and aperiodic components

 Look at any EEG power spectrum and you'll notice two things:
1. **Power decreases with frequency** — a broad downward slope (the "1/f" or aperiodic component).
2. **Peaks sit on top of that slope** — these are the oscillations (alpha, beta, etc.).

Standard spectral analysis mixes these two together. If the 1/f slope is steep, low-frequency power looks huge even without any oscillation. And if you compare spectra between conditions or groups, a change in the slope can masquerade as a change in oscillatory power.

**specparam** (formerly called FOOOF — "Fitting Oscillations & One-Over-F") solves this by fitting a parametric model to the spectrum:

 `Power(f) = Aperiodic(f) + Peaks(f)`

 - Aperiodic: `offset − exponent · log(f)` — the 1/f background
 - Peaks: Gaussians centered at specific frequencies — the oscillations

 It returns the parameters of both, cleanly separated.

 Let's first see what a 1/f spectrum looks like, and what happens when we add oscillations on top.

In [ ]:
fs = 500
duration = 30  
t_fooof = np.arange(0, duration, 1 / fs)
np.random.seed(42)
 

white = np.random.randn(len(t_fooof))
freqs_shape = np.fft.rfftfreq(len(t_fooof), 1 / fs)
freqs_shape[0] = freqs_shape[1]  
pink_filter = 1 / np.sqrt(freqs_shape)
pink = np.fft.irfft(np.fft.rfft(white) * pink_filter, n=len(t_fooof))
pink = pink / np.std(pink) * 1.5  # scale
 

alpha_osc = 1.5 * np.sin(2 * np.pi * 10 * t_fooof)
beta_osc  = 0.5 * np.sin(2 * np.pi * 22 * t_fooof)
 
eeg_like = pink + alpha_osc + beta_osc

f_pink, psd_pink = scipy_welch(pink, fs=fs, nperseg=2 * fs)
f_eeg,  psd_eeg  = scipy_welch(eeg_like, fs=fs, nperseg=2 * fs)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
axes[0].loglog(f_pink, psd_pink, color='gray', linewidth=1.5)
axes[0].set_title('Pure 1/f noise — no oscillations')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Power')
axes[0].set_xlim(1, 50)
 
axes[1].loglog(f_eeg, psd_eeg, color='steelblue', linewidth=1.5)
axes[1].set_title('1/f + alpha (10 Hz) + beta (22 Hz)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_xlim(1, 50)
axes[1].annotate('alpha', xy=(10, psd_eeg[np.argmin(np.abs(f_eeg - 10))]),
                 fontsize=11, ha='left', color='crimson', fontweight='bold')
axes[1].annotate('beta', xy=(22, psd_eeg[np.argmin(np.abs(f_eeg - 22))]),
                 fontsize=11, ha='left', color='crimson', fontweight='bold')
 
plt.suptitle('EEG-like spectrum = 1/f slope + oscillatory peaks on top',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

On a log-log scale, the 1/f background is a straight line. The oscillations appear as bumps above this line. But notice how on a regular (linear) scale the 1/f slope makes the low frequencies look much "bigger" — this is why you should always think about what's an oscillation and what's just the background.

In [ ]:
# ! pip3 install specparam

In [ ]:
import specparam
 
sm = specparam.SpectralModel(
    peak_width_limits=[1, 8],   # peaks must be 1–8 Hz wide
    max_n_peaks=6,              # look for up to 6 peaks
    min_peak_height=0.1,        # ignore tiny bumps
    aperiodic_mode='fixed',     # simple 1/f (no knee)
)
sm.fit(f_eeg, psd_eeg, [1, 45])
 

ap_params = sm.get_params('aperiodic')  # [offset, exponent]
peak_params = sm.get_params('peak')     # [[center_freq, power, bandwidth], ...]
r_squared = sm.results.get_results().metrics['gof_rsquared']
 
print(f'Aperiodic: offset = {ap_params[0]:.2f}, exponent = {ap_params[1]:.2f}')
print(f'Peaks found:')
for p in peak_params:
    print(f'  {p[0]:.1f} Hz — power = {p[1]:.2f}, bandwidth = {p[2]:.1f} Hz')
print(f'Goodness of fit (R²): {r_squared:.3f}')

specparam found:
- The aperiodic exponent — the steepness of the 1/f slope.
- The oscillatory peaks — alpha at ~10 Hz and beta at ~22 Hz.
- How well the model fits the data (R²).

Now let's visualize the decomposition.

In [ ]:
freqs_fit = sm.data.freqs
power_log = np.log10(sm.data.power_spectrum)  # original data
model_fit = sm.results.model.modeled_spectrum  # full model
ap_fit    = sm.results.model._ap_fit           # aperiodic fit only
flat_spec = sm.results.model._spectrum_flat    # data minus aperiodic (flattened)
peak_fit  = sm.results.model._peak_fit         # Gaussian peaks only
 
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
 

axes[0].plot(freqs_fit, power_log, 'k', linewidth=1.5, alpha=0.7, label='Data')
axes[0].plot(freqs_fit, model_fit, 'r-', linewidth=2, label='Full model')
axes[0].plot(freqs_fit, ap_fit, 'b--', linewidth=1.5, label='Aperiodic fit')
axes[0].set_title('Step 1: Fit aperiodic + peaks')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('log₁₀(Power)')
axes[0].legend(fontsize=9)
 

axes[1].plot(freqs_fit, flat_spec, 'k', linewidth=1.5, alpha=0.7, label='Flattened data')
axes[1].plot(freqs_fit, peak_fit, 'r-', linewidth=2, label='Gaussian peaks')
axes[1].axhline(0, color='gray', linewidth=0.5, linestyle=':')
axes[1].set_title('Step 2: Remove 1/f → only oscillations remain')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('log₁₀(Power) − aperiodic')
axes[1].legend(fontsize=9)
 

spec_no_peaks = sm.results.model._spectrum_peak_rm
axes[2].plot(freqs_fit, spec_no_peaks, 'k', linewidth=1.5, alpha=0.7,
             label='Peaks removed')
axes[2].plot(freqs_fit, ap_fit, 'b--', linewidth=1.5, label='Aperiodic fit')
axes[2].set_title('Step 3: Remove peaks → only 1/f remains')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel('log₁₀(Power)')
axes[2].legend(fontsize=9)
 
plt.suptitle('specparam decomposition: aperiodic (1/f) + periodic (peaks)',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

 Panel 1: The red line is the full model. The blue dashed line is just the aperiodic (1/f) fit. The peaks sit above the blue line.

Panel 2: After subtracting the 1/f component, only the oscillatory peaks remain. This is the "flattened" spectrum — what you should use when you want to measure oscillatory power cleanly.

Panel 3: After subtracting the peaks, only the smooth 1/f background remains. This gives you the aperiodic exponent, which has been linked to the excitation/inhibition balance in neural circuits.

 Why does this matter? If you just compare "alpha power" between two groups by taking the raw spectrum in 8–12 Hz, you may actually be measuring differences in the 1/f slope, not in oscillations. Let's demonstrate: two signals with the same oscillation but different 1/f exponents.

In [ ]:
np.random.seed(99)
white1 = np.random.randn(len(t_fooof))
white2 = np.random.randn(len(t_fooof))
 
# Shallow 1/f (exponent ~ 1)
pink1 = np.fft.irfft(np.fft.rfft(white1) * pink_filter, n=len(t_fooof))
pink1 = pink1 / np.std(pink1) * 1.5
 
# Steep 1/f (exponent ~ 2)
steep_filter = 1 / freqs_shape  # 1/f instead of 1/sqrt(f)
pink2 = np.fft.irfft(np.fft.rfft(white2) * steep_filter, n=len(t_fooof))
pink2 = pink2 / np.std(pink2) * 1.5
 
# Same alpha oscillation in both
alpha_same = 1.0 * np.sin(2 * np.pi * 10 * t_fooof)
 
signal1 = pink1 + alpha_same
signal2 = pink2 + alpha_same
 
f1, psd1 = scipy_welch(signal1, fs=fs, nperseg=2 * fs)
f2, psd2 = scipy_welch(signal2, fs=fs, nperseg=2 * fs)
 

sm1 = specparam.SpectralModel(peak_width_limits=[1, 8], max_n_peaks=4,
                               min_peak_height=0.1, aperiodic_mode='fixed')
sm2 = specparam.SpectralModel(peak_width_limits=[1, 8], max_n_peaks=4,
                               min_peak_height=0.1, aperiodic_mode='fixed')
sm1.fit(f1, psd1, [1, 45])
sm2.fit(f2, psd2, [1, 45])
 
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
 

axes[0].semilogy(f1, psd1, color='steelblue', linewidth=1.5, label='Signal A')
axes[0].semilogy(f2, psd2, color='crimson', linewidth=1.5, label='Signal B')
axes[0].axvspan(8, 12, alpha=0.2, color='green', label='Alpha band')
axes[0].set_title('Raw spectra — B looks like "more alpha"')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Power (log scale)')
axes[0].set_xlim(1, 45)
axes[0].legend(fontsize=9)
 

alpha_mask = (f1 >= 8) & (f1 <= 12)
alpha_pow1 = np.sum(psd1[alpha_mask])
alpha_pow2 = np.sum(psd2[alpha_mask])
axes[1].bar(['Signal A', 'Signal B'], [alpha_pow1, alpha_pow2],
            color=['steelblue', 'crimson'], alpha=0.7)
axes[1].set_title('Naive alpha power (sum 8–12 Hz)')
axes[1].set_ylabel('Power')
axes[1].annotate(f'{alpha_pow2/alpha_pow1:.1f}× more!', xy=(1, alpha_pow2),
                 fontsize=12, ha='center', va='bottom', color='red', fontweight='bold')
 

peaks1 = sm1.get_params('peak')
peaks2 = sm2.get_params('peak')

def find_alpha_peak(peaks):
    if peaks.ndim == 1:
        peaks = peaks[np.newaxis, :]
    alpha_peaks = peaks[(peaks[:, 0] >= 8) & (peaks[:, 0] <= 12)]
    return alpha_peaks[0, 1] if len(alpha_peaks) > 0 else 0
 
ap1_power = find_alpha_peak(peaks1)
ap2_power = find_alpha_peak(peaks2)
exp1 = sm1.get_params('aperiodic')[1]
exp2 = sm2.get_params('aperiodic')[1]
 
axes[2].bar(['Signal A', 'Signal B'], [ap1_power, ap2_power],
            color=['steelblue', 'crimson'], alpha=0.7)
axes[2].set_title('specparam alpha peak power\n(aperiodic removed)')
axes[2].set_ylabel('Peak power (a.u.)')
axes[2].annotate(f'exp={exp1:.1f}', xy=(0, ap1_power), fontsize=10,
                 ha='center', va='bottom', color='steelblue')
axes[2].annotate(f'exp={exp2:.1f}', xy=(1, ap2_power), fontsize=10,
                 ha='center', va='bottom', color='crimson')
 
plt.suptitle('Same oscillation, different 1/f → naive measure is misleading',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()